# MiniMax AutoResearch Chess Workshop

This notebook walks through the demo loop: baseline chess bot, local estimated Elo evaluator, constrained MiniMax patch, accept/reject, and replay.

## 1. Baseline

The starting bot is intentionally weak but legal. It uses `python-chess` for move legality and a shallow material-heavy search.

In [ ]:
!python -m autoresearch_chess.eval --out artifacts/notebook_eval.json

## 2. Editable Surface

MiniMax may edit only `bot/evaluate.py`, `bot/search.py`, `bot/move_ordering.py`, and `bot/config.py`. The evaluator, opponents, artifacts, tests, env files, and docs are forbidden during eval.

In [ ]:
!python -m autoresearch_chess.loop --stage --mock-minimax --iterations 2

## 3. Replay

If live MiniMax or network access fails on stage, replay the captured artifact run.

In [ ]:
!python -m autoresearch_chess.replay --run artifacts/demo_replay

## 4. Tool Calling in Action

Each iteration now drives an OpenClaw-style ReAct loop: MiniMax inspects the editable bot files and recent history via tool calls, then submits a final unified diff with the terminal `propose_patch` tool. The full trace is persisted to `iterations/NNN/agent_trace.jsonl` so attendees can read exactly what the agent did.

See `docs/openclaw_mapping.md` for how the agent modules map onto OpenClaw's gateway / context / react / tool-layer architecture.

In [ ]:
from pathlib import Path
import json

runs = sorted(Path('artifacts/runs').glob('*/iterations/001/agent_trace.jsonl'), key=lambda p: p.stat().st_mtime)
trace_path = runs[-1]
print('Reading', trace_path)
for line in trace_path.read_text(encoding='utf-8').splitlines():
    event = json.loads(line)
    calls = ', '.join(tc['name'] for tc in event['tool_calls']) or '<final message>'
    print(f"round {event['round']}: {calls}")

**Try this:** open the JSONL file in your editor and read the `arguments` and `result_preview` fields for each tool call. That is the agent's reasoning, made inspectable.

**Try this:** edit a tool description in `autoresearch_chess/agent/tools.py`, re-run the loop, and observe how the trace changes.